# ChiFraud 簡繁雙模型實驗（Kaggle）

> 文件版本：v1.1｜日期：2026-07-23｜作者：ScamLens-TW 專案團隊  
> 摘要：固定官方來源與 base model revision，呼叫 repo 內正式核心完成簡繁訓練、交叉驗收與 artifact 匯出。

```mermaid
flowchart LR
    S[官方 ChiFraud commit] --> P[成對資料準備] --> M[簡體與繁體 BERT] --> E[跨年度／字體驗收] --> A[可下載 artifacts]
```

2022 與 2023 都會進入 train／validation／test；test 不參與 checkpoint、溫度或門檻選擇。資料不重新發布，直接從 [ChiFraud 官方 repository](https://github.com/xuemingxxx/ChiFraud) 取得；base model 來自 [Google bert-base-chinese](https://huggingface.co/google-bert/bert-base-chinese)，兩者於 2026-07-23 查核。

In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT_ROOT = Path('/kaggle/working/AI_Voice')
if not (PROJECT_ROOT / 'src').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/EYY7592/AI_Voice.git', str(PROJECT_ROOT)], check=True)
elif (Path.cwd() / 'src').is_dir():
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)
PROJECT_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print({'project_revision': PROJECT_REVISION, 'project_root': str(PROJECT_ROOT)})
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import importlib.metadata as metadata, platform, torch
print({'python': platform.python_version(), 'platform': platform.platform(), 'gpu_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
if not torch.cuda.is_available():
    raise RuntimeError('此完整實驗要求 Kaggle GPU accelerator。')
for package in ('torch', 'transformers', 'opencc-python-reimplemented', 'safetensors'):
    print(package, metadata.version(package))
print({'base_model_revision': '8f23c25b06e129b6c986331a13d8d025a92cf0ea', 'data_revision': '5a0245743e85154f114f2b3ea24b0bed4e64cf0c', 'split_seed': 42})


In [ ]:
CHIFRAUD_REVISION = '5a0245743e85154f114f2b3ea24b0bed4e64cf0c'
BERT_REVISION = '8f23c25b06e129b6c986331a13d8d025a92cf0ea'
source_dir = Path('/kaggle/working/ChiFraud')
if not source_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/xuemingxxx/ChiFraud.git', str(source_dir)], check=True)
subprocess.run(['git', '-C', str(source_dir), 'checkout', '--detach', CHIFRAUD_REVISION], check=True)
subprocess.run([sys.executable, '-m', 'src.chifraud_data', str(source_dir / 'dataset'), '/kaggle/working/chifraud_prepared', '--source-revision', CHIFRAUD_REVISION, '--seed', '42'], check=True)


In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.run_chifraud_experiment',
    '/kaggle/working/chifraud_prepared', '/kaggle/working/chifraud_experiment',
    '--base-model', 'google-bert/bert-base-chinese',
    '--base-revision', BERT_REVISION,
    '--seeds', '42',
    '--max-epochs', '8', '--patience', '2', '--batch-size', '16'
], check=True)


In [ ]:
import json
report_path = Path('/kaggle/working/chifraud_experiment/selection_report.json')
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report['selection'], ensure_ascii=False, indent=2))
print('若 status=additional_seeds_required，請在全新 output 以 --seeds 42 43 44 重跑；只有 status=selected 且候選通過驗收才可下載升級。')
import shutil
import matplotlib.pyplot as plt
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for manifest in report['candidate_manifests']:
    label = f"{manifest['script_view']}-seed-{manifest['seed']}"
    epochs = [item['epoch'] for item in manifest['history']]
    axes[0].plot(epochs, [item['train_loss'] for item in manifest['history']], label=label)
    axes[1].plot(epochs, [item['validation_loss'] for item in manifest['history']], label=label)
axes[0].set_title('Training loss'); axes[1].set_title('Validation loss')
for axis in axes: axis.set_xlabel('epoch'); axis.legend()
figure.tight_layout(); figure.savefig('/kaggle/working/chifraud_experiment/learning_curves.png', dpi=150)
plt.show()
archive = shutil.make_archive('/kaggle/working/chifraud_experiment', 'zip', '/kaggle/working/chifraud_experiment')
print('可下載 artifact：', archive)
